In [16]:
%pip install langchain_huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
%pip install unstructured

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------- ----------------------------- 262.1/981.5 kB ? eta -:--:--
     ------------------------------ ------- 786.4/981.5 kB 1.8 MB/s eta 0:00:01
     -------------------------------------- 981.5/981.5 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached webencodings-0.5.1-py2.py3-none-any.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   --


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install -qU langchain-community beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
from langchain_community.document_loaders import UnstructuredHTMLLoader, UnstructuredMarkdownLoader, TextLoader, BSHTMLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from pathlib import Path

In [34]:
base_dir = Path("../../docs")
paths = list(base_dir.rglob("*"))

def load_docs(paths):
    all_docs = []
    for p in paths:
        if not p.is_file():
            continue

        ext = p.suffix.lower()
        try:
            if ext == ".html":
                # Try UnstructuredHTMLLoader first, fallback to BSHTMLLoader if it fails
                try:
                    loader = UnstructuredHTMLLoader(p)
                    docs = loader.load()
                except (AttributeError, Exception) as e:
                    # Fallback to BSHTMLLoader for problematic HTML files
                    print(f"Warning: UnstructuredHTMLLoader failed for {p}, using BSHTMLLoader instead. Error: {type(e).__name__}")
                    loader = BSHTMLLoader(p)
                    docs = loader.load()
            elif ext == ".md":
                loader = UnstructuredMarkdownLoader(p)
                docs = loader.load()
            elif ext == ".txt":
                loader = TextLoader(p)
                docs = loader.load()
            else:
                print(f"Skipping {p} because it is not a supported file type")
                continue
            
            all_docs.extend(docs)
        except Exception as e:
            print(f"Error loading {p}: {type(e).__name__}: {e}")
            continue
    
    return all_docs

print(load_docs(paths))


[Document(metadata={'source': '..\\..\\docs\\abc.html'}, page_content='Navigation\n\nindex\n\nmodules |\n\nnext |\n\nprevious |\n\nPython logo\n\nPython »\n\n3.15.0a6 Documentation »\n\nThe Python Standard Library »\n\nPython Runtime Services »\n\nabc — Abstract Base Classes\n\n|\n\nabc — Abstract Base Classes¶\n\nSource code: Lib/abc.py\n\nThis module provides the infrastructure for defining abstract base classes (ABCs) in Python, as outlined in PEP 3119; see the PEP for why this was added to Python. (See also PEP 3141 and the numbers module regarding a type hierarchy for numbers based on ABCs.)\n\nThe collections module has some concrete classes that derive from ABCs; these can, of course, be further derived. In addition, the collections.abc submodule has some ABCs that can be used to test whether a class or instance provides a particular interface, for example, if it is hashable or if it is a mapping.\n\nThis module provides the metaclass ABCMeta for defining ABCs and a helper class

In [35]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
texts = splitter.split_documents(load_docs(paths))
print(texts[0])

page_content='Navigation

index

modules |

next |

previous |

Python logo

Python »

3.15.0a6 Documentation »

The Python Standard Library »

Python Runtime Services »

abc — Abstract Base Classes

|

abc — Abstract Base Classes¶

Source code: Lib/abc.py

This module provides the infrastructure for defining abstract base classes (ABCs) in Python, as outlined in PEP 3119; see the PEP for why this was added to Python. (See also PEP 3141 and the numbers module regarding a type hierarchy for numbers based on ABCs.)

The collections module has some concrete classes that derive from ABCs; these can, of course, be further derived. In addition, the collections.abc submodule has some ABCs that can be used to test whether a class or instance provides a particular interface, for example, if it is hashable or if it is a mapping.

This module provides the metaclass ABCMeta for defining ABCs and a helper class ABC to alternatively define ABCs through inheritance:

The abc module also provides the 

In [37]:
print(texts[1])

page_content='Next topic

atexit — Exit handlers

This page

Report a bug

Improve this page

Show source

Navigation

index

modules |

next |

previous |

Python logo

Python »

3.15.0a6 Documentation »

The Python Standard Library »

Python Runtime Services »

abc — Abstract Base Classes

|

© Copyright 2001 Python Software Foundation. This page is licensed under the Python Software Foundation License Version 2. Examples, recipes, and other code in the documentation are additionally licensed under the Zero Clause BSD License. See History and License for more information. The Python Software Foundation is a non-profit corporation. Please donate. Last updated on Mar 10, 2026 (08:58 UTC). Found a bug? Created using Sphinx 8.2.3.' metadata={'source': '..\\..\\docs\\abc.html'}


In [38]:
model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Chroma.from_documents() expects the embedding MODEL, not pre-computed embeddings
# It will automatically extract text from Document objects and embed them
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=model,  # Pass the model, not embeddings
    persist_directory="chroma_db",
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2848.21it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
